# Experiment 2: Mixed Training (HAM10000 + PAD-UFES-20)

Train MedSigLIP on both dermoscopic (HAM10000) and clinical smartphone (PAD-UFES-20) images to bridge the domain gap.

**Key techniques:** 3x PAD-UFES oversampling, Focal Loss, 2x melanoma class weight, stronger augmentation.

**Results:** PAD-UFES balanced accuracy improved from 49.9% to **81.8%** (+31.9 points).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kabirrgrover/dermtriage-public/blob/master/notebooks/02_train_mixed.ipynb)

In [ ]:
# Install dependencies & authenticate
!pip install -q torch torchvision transformers scikit-learn tqdm kaggle huggingface-hub pandas

import os
from google.colab import userdata, drive
from huggingface_hub import login

login(token=userdata.get('HF_TOKEN'))
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

# Mount Drive for checkpoint saving
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

In [ ]:
# Download datasets
import subprocess, shutil
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

CLASS_NAMES = ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']
PADUFES_TO_HAM = {'ACK': 'akiec', 'BCC': 'bcc', 'MEL': 'mel', 'NEV': 'nv', 'SEK': 'bkl'}

# HAM10000
DATA_DIR = Path('/content/data')
PROCESSED_DIR = DATA_DIR / 'processed'

if not (PROCESSED_DIR / 'train').exists():
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    subprocess.run(['kaggle', 'datasets', 'download', '-d', 'kmader/skin-cancer-mnist-ham10000',
                    '-p', str(DATA_DIR), '--unzip'], check=True)
    df = pd.read_csv(DATA_DIR / 'HAM10000_metadata.csv')
    img_dirs = [DATA_DIR / 'HAM10000_images_part_1', DATA_DIR / 'HAM10000_images_part_2']
    for split in ['train', 'val']:
        for cls in CLASS_NAMES:
            (PROCESSED_DIR / split / cls).mkdir(parents=True, exist_ok=True)
    train_df, val_df = train_test_split(df, test_size=0.2, stratify=df['dx'], random_state=42)
    for split_df, split_name in [(train_df, 'train'), (val_df, 'val')]:
        for _, row in split_df.iterrows():
            img_name = f"{row['image_id']}.jpg"
            for img_dir in img_dirs:
                src = img_dir / img_name
                if src.exists():
                    shutil.copy(src, PROCESSED_DIR / split_name / row['dx'] / img_name)
                    break
    print('HAM10000 ready!')
else:
    print('HAM10000 already prepared!')

# PAD-UFES-20
PADUFES_DIR = Path('/content/padufes')
if not PADUFES_DIR.exists():
    subprocess.run(['kaggle', 'datasets', 'download', '-d', 'mahdavi1202/skin-cancer',
                    '-p', str(PADUFES_DIR), '--unzip'], check=True)
    print('PAD-UFES-20 downloaded!')
else:
    print('PAD-UFES-20 already exists!')

In [ ]:
# Imports
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, ConcatDataset, WeightedRandomSampler
from torchvision import transforms
from PIL import Image
from tqdm import tqdm
import numpy as np
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, recall_score, precision_score

NUM_CLASSES = len(CLASS_NAMES)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
# Dataset classes
class HAM10000Dataset(Dataset):
    def __init__(self, data_dir, split='train', transform=None):
        self.transform = transform
        self.class_to_idx = {name: idx for idx, name in enumerate(CLASS_NAMES)}
        self.samples = []
        split_dir = Path(data_dir) / split
        for class_name in CLASS_NAMES:
            class_dir = split_dir / class_name
            if class_dir.exists():
                for img_path in class_dir.glob('*.jpg'):
                    self.samples.append({'path': img_path, 'label': self.class_to_idx[class_name], 'class_name': class_name})
        print(f'HAM10000 {split}: {len(self.samples)} samples')

    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        s = self.samples[idx]
        image = Image.open(s['path']).convert('RGB')
        if self.transform: image = self.transform(image)
        return image, s['label']


class PADUFESDataset(Dataset):
    def __init__(self, data_dir, split='train', transform=None, val_ratio=0.2):
        self.transform = transform
        self.class_to_idx = {name: idx for idx, name in enumerate(CLASS_NAMES)}
        self.data_dir = Path(data_dir)
        metadata_path = next(self.data_dir.rglob('*.csv'), None)
        if metadata_path is None: raise FileNotFoundError(f'No CSV in {data_dir}')
        df = pd.read_csv(metadata_path)
        diag_col = 'diagnostic' if 'diagnostic' in df.columns else 'dx'
        df = df[df[diag_col].isin(PADUFES_TO_HAM.keys())].copy()
        df['ham_class'] = df[diag_col].map(PADUFES_TO_HAM)
        df['label'] = df['ham_class'].map(self.class_to_idx)
        train_df, val_df = train_test_split(df, test_size=val_ratio, stratify=df['ham_class'], random_state=42)
        self.df = train_df if split == 'train' else val_df
        self.img_dirs = [d for d in self.data_dir.rglob('*') if d.is_dir() and len(list(d.glob('*.png'))) > 10]
        img_id_col = 'img_id' if 'img_id' in df.columns else 'image_id'
        fitz_col = 'fitspatrick' if 'fitspatrick' in df.columns else None
        self.samples = [{'img_id': row[img_id_col], 'label': row['label'], 'class_name': row['ham_class'],
                         'fitzpatrick': row[fitz_col] if fitz_col else None} for _, row in self.df.iterrows()]
        print(f'PAD-UFES-20 {split}: {len(self.samples)} samples')

    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        s = self.samples[idx]
        img_id_base = s['img_id'].rsplit('.', 1)[0] if s['img_id'].endswith(('.png', '.jpg')) else s['img_id']
        img_path = None
        for d in self.img_dirs:
            for ext in ['.png', '.PNG', '.jpg']:
                c = d / f'{img_id_base}{ext}'
                if c.exists(): img_path = c; break
            if not img_path:
                c = d / s['img_id']
                if c.exists(): img_path = c
            if img_path: break
        if img_path is None: raise FileNotFoundError(f'Image not found: {s["img_id"]}')
        image = Image.open(img_path).convert('RGB')
        if self.transform: image = self.transform(image)
        return image, s['label']

    def get_fitzpatrick_groups(self):
        groups = {'I-II': [], 'III-IV': [], 'V-VI': [], 'unknown': []}
        for idx, s in enumerate(self.samples):
            f = s['fitzpatrick']
            if f in [1, 2]: groups['I-II'].append(idx)
            elif f in [3, 4]: groups['III-IV'].append(idx)
            elif f in [5, 6]: groups['V-VI'].append(idx)
            else: groups['unknown'].append(idx)
        return groups

print('Dataset classes defined!')

In [ ]:
# Model & Loss
class MedSigLIPClassifier(nn.Module):
    def __init__(self, num_classes=7, dropout_rate=0.3):
        super().__init__()
        from transformers import AutoModel
        full_model = AutoModel.from_pretrained('google/medsiglip-448', torch_dtype=torch.float32)
        self.vision_model = full_model.vision_model
        self.embed_dim = full_model.config.vision_config.hidden_size
        del full_model
        for param in self.vision_model.parameters(): param.requires_grad = False
        self.classifier = nn.Sequential(
            nn.LayerNorm(self.embed_dim), nn.Dropout(dropout_rate),
            nn.Linear(self.embed_dim, 512), nn.GELU(), nn.Dropout(dropout_rate),
            nn.Linear(512, num_classes)
        )

    def forward(self, pixel_values):
        with torch.no_grad():
            outputs = self.vision_model(pixel_values=pixel_values)
            features = outputs.pooler_output if hasattr(outputs, 'pooler_output') and outputs.pooler_output is not None else outputs.last_hidden_state.mean(dim=1)
        return self.classifier(features)


class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha
    def forward(self, inputs, targets):
        ce_loss = nn.functional.cross_entropy(inputs, targets, weight=self.alpha, reduction='none')
        pt = torch.exp(-ce_loss)
        return (((1 - pt) ** self.gamma) * ce_loss).mean()


def evaluate_model(model, dataloader, dataset_name=''):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc=f'Eval {dataset_name}'):
            preds = torch.argmax(model(images.to(device)), dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())
    all_preds, all_labels = np.array(all_preds), np.array(all_labels)
    mel_idx = CLASS_NAMES.index('mel')
    return {
        'balanced_accuracy': balanced_accuracy_score(all_labels, all_preds),
        'macro_f1': f1_score(all_labels, all_preds, average='macro', zero_division=0),
        'melanoma_recall': recall_score(all_labels, all_preds, labels=[mel_idx], average='micro', zero_division=0),
    }

print('Model and evaluation ready!')

In [ ]:
# Create data loaders
val_transform = transforms.Compose([
    transforms.Resize((448, 448)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])
train_transform = transforms.Compose([
    transforms.Resize((448, 448)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.GaussianBlur(kernel_size=5, sigma=(0.1, 2.0)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

ham_train = HAM10000Dataset(str(PROCESSED_DIR), split='train', transform=train_transform)
pad_train = PADUFESDataset(str(PADUFES_DIR), split='train', transform=train_transform)
combined = ConcatDataset([ham_train, pad_train])
weights = [1.0]*len(ham_train) + [3.0]*len(pad_train)
sampler = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)
train_loader = DataLoader(combined, batch_size=32, sampler=sampler, num_workers=2)

ham_val = HAM10000Dataset(str(PROCESSED_DIR), split='val', transform=val_transform)
pad_val = PADUFESDataset(str(PADUFES_DIR), split='val', transform=val_transform)
ham_val_loader = DataLoader(ham_val, batch_size=16, shuffle=False, num_workers=2)
pad_val_loader = DataLoader(pad_val, batch_size=16, shuffle=False, num_workers=2)

print(f'Training: {len(combined)} samples (HAM: {len(ham_train)}, PAD: {len(pad_train)} x3 weighted)')

In [ ]:
# Training loop
model = MedSigLIPClassifier(num_classes=NUM_CLASSES).to(device)

class_wts = torch.ones(NUM_CLASSES).to(device)
class_wts[CLASS_NAMES.index('mel')] = 2.0
criterion = FocalLoss(alpha=class_wts, gamma=2.0)
optimizer = optim.AdamW(model.classifier.parameters(), lr=1e-3, weight_decay=0.01)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30)

PATIENCE = 10
best_score, patience_counter, best_epoch = 0, 0, 0
CHECKPOINT_DIR = Path('/content/checkpoints')
CHECKPOINT_DIR.mkdir(exist_ok=True)

print('=' * 60)
print('EXPERIMENT 2: MIXED TRAINING')
print('Scoring: 0.4*HAM_bal_acc + 0.3*PAD_bal_acc + 0.3*avg_mel_recall')
print('=' * 60)

for epoch in range(30):
    model.train()
    train_loss, train_correct, train_total = 0, 0, 0
    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/30')
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        _, predicted = outputs.max(1)
        train_total += labels.size(0)
        train_correct += predicted.eq(labels).sum().item()
        pbar.set_postfix({'loss': f'{loss.item():.4f}', 'acc': f'{100.*train_correct/train_total:.1f}%'})
    scheduler.step()

    ham_r = evaluate_model(model, ham_val_loader, 'HAM10000')
    pad_r = evaluate_model(model, pad_val_loader, 'PAD-UFES-20')
    mel_avg = (ham_r['melanoma_recall'] + pad_r['melanoma_recall']) / 2
    score = 0.4*ham_r['balanced_accuracy'] + 0.3*pad_r['balanced_accuracy'] + 0.3*mel_avg

    print(f'Epoch {epoch+1}: Score={score:.4f}')
    print(f"  HAM: Bal_Acc={ham_r['balanced_accuracy']:.3f}, Mel_Recall={ham_r['melanoma_recall']:.3f}")
    print(f"  PAD: Bal_Acc={pad_r['balanced_accuracy']:.3f}, Mel_Recall={pad_r['melanoma_recall']:.3f}")

    if score > best_score:
        best_score, patience_counter, best_epoch = score, 0, epoch + 1
        torch.save({'epoch': epoch+1, 'model_state_dict': model.state_dict(),
                    'combined_score': score, 'ham_results': ham_r, 'pad_results': pad_r},
                   CHECKPOINT_DIR / 'best_model_mixed.pth')
        print('  ** New best! Saved. **')
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f'Early stopping at epoch {epoch+1}')
            break

print(f'\nDone! Best epoch: {best_epoch}, Best score: {best_score:.4f}')

## Results (Experiment 2)

| Dataset | Metric | Exp 1 (HAM only) | Exp 2 (Mixed) | Delta |
|---------|--------|-------------------|---------------|-------|
| HAM10000 | Balanced Accuracy | 78.9% | 69.6% | -9.3 |
| HAM10000 | Melanoma Recall | 85.7% | 77.6% | -8.1 |
| PAD-UFES-20 | Balanced Accuracy | 49.9% | **81.8%** | **+31.9** |
| PAD-UFES-20 | Melanoma Recall | 54.5% | **81.8%** | **+27.3** |

### Fairness (PAD-UFES-20)

| Fitzpatrick | Mel Recall (Exp 1) | Mel Recall (Exp 2) |
|-------------|--------------------|-----------------------|
| I-II | 50.0% | 75.0% |
| III-IV | 66.7% | **100.0%** |

Mixed training improved performance for darker skin tones (III-IV: 100% melanoma recall).